In [ ]:
!pip -q install optuna

In [ ]:
!pip install optuna-integration[tfkeras]

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import mixed_precision
import optuna
from optuna.integration import TFKerasPruningCallback
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize

In [ ]:
from google.colab import files
files.upload()


In [ ]:
!unzip -q "HAM_CLEAN_SPLIT.zip" -d /content/dataset

In [ ]:
CLEAN_ROOT = "/content/dataset/HAM_CLEAN_SPLIT"
TRAIN_DIR  = os.path.join(CLEAN_ROOT, "train")
VAL_DIR    = os.path.join(CLEAN_ROOT, "val")
TEST_DIR   = os.path.join(CLEAN_ROOT, "test")

IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("Classes:", class_names)

In [ ]:
mixed_precision.set_global_policy("mixed_float16")

AUTOTUNE = tf.data.AUTOTUNE
preprocess = tf.keras.applications.mobilenet_v2.preprocess_input

def preprocess_batch(x, y):
    x = tf.cast(x, tf.float32)
    x = preprocess(x)
    return x, y

train_ds = train_ds.map(preprocess_batch, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_ds.map(preprocess_batch,   num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds  = test_ds.map(preprocess_batch,  num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

In [ ]:


def build_baseline_model(dropout):

    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights="imagenet"
    )
    base_model.trainable = False

    inputs = keras.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)

    model = keras.Model(inputs, outputs)
    return model

In [ ]:
def objective(trial):

    lr_head = trial.suggest_float("lr_head", 3e-5, 3e-3, log=True)
    lr_ft   = trial.suggest_float("lr_ft", 1e-6, 3e-4, log=True)
    dropout = trial.suggest_float("dropout", 0.2, 0.6, step=0.05)
    unfreeze_last = trial.suggest_categorical("unfreeze_last", [20,30, 40, 50,60])
    weight_decay = trial.suggest_categorical("weight_decay", [0.0, 1e-5, 1e-4])
    label_smoothing = trial.suggest_categorical("label_smoothing", [0.0, 0.05, 0.1])

    model = build_baseline_model(dropout)

    loss_fn = keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr_head, weight_decay=weight_decay),
        loss=loss_fn,
        metrics=["accuracy"]
    )

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=6,
        callbacks=[
            keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True),
            TFKerasPruningCallback(trial, "val_accuracy")
        ],
        verbose=0
    )

    
    base_model = next(
        l for l in model.layers
        if isinstance(l, tf.keras.Model) and "mobilenetv2" in l.name.lower()
    )

    base_model.trainable = True

    for layer in base_model.layers[:-unfreeze_last]:
        layer.trainable = False

    for layer in base_model.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr_ft, weight_decay=weight_decay),
        loss=loss_fn,
        metrics=["accuracy"]
    )

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10,
        callbacks=[
            keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
            TFKerasPruningCallback(trial, "val_accuracy")
        ],
        verbose=0
    )

    return float(max(history.history["val_accuracy"]))

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)  

In [ ]:
print("Best Validation Accuracy:", study.best_value)
print("Best Hyperparameters:")
for k, v in study.best_params.items():
    print(k, ":", v)

In [ ]:
best_params = study.best_params

print("Training final model with best parameters...")
print(best_params)

final_model = build_baseline_model(best_params["dropout"])

loss_fn = keras.losses.CategoricalCrossentropy(
    label_smoothing=best_params["label_smoothing"]
)
ckpt_path = "mobilenetv2_tuned_best.keras"


final_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=best_params["lr_head"],
        weight_decay=best_params["weight_decay"]
    ),
    loss=loss_fn,
    metrics=["accuracy"]
)

callbacks_fe = [
     keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True),
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=4,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2
    )
]

history_fe = final_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks_fe,
    verbose=1
)



base_model = next(
    l for l in final_model.layers
    if isinstance(l, tf.keras.Model) and "mobilenetv2" in l.name.lower()
)

base_model.trainable = True

for layer in base_model.layers[:-best_params["unfreeze_last"]]:
    layer.trainable = False

for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

final_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=best_params["lr_ft"],
        weight_decay=best_params["weight_decay"]
    ),
    loss=loss_fn,
    metrics=["accuracy"]
)

callbacks_ft = [
    keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True),
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2
    )
]

history_ft = final_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks_ft,
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

def combine_histories(h1, h2):
    combined = {}
    for key in ["accuracy", "val_accuracy", "loss", "val_loss"]:
        combined[key] = h1.history.get(key, []) + h2.history.get(key, [])
    return combined

combined = combine_histories(history_fe, history_ft)

epochs_total = range(1, len(combined["loss"]) + 1)
split_epoch = len(history_fe.history["loss"])


plt.figure(figsize=(8,5))
plt.plot(epochs_total, combined["accuracy"], label="Train Accuracy")
plt.plot(epochs_total, combined["val_accuracy"], label="Val Accuracy")
plt.axvline(split_epoch, linestyle="--", color="black", label="Start Fine-Tuning")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training & Validation Accuracy (Tuned Baseline)")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8,5))
plt.plot(epochs_total, combined["loss"], label="Train Loss")
plt.plot(epochs_total, combined["val_loss"], label="Val Loss")
plt.axvline(split_epoch, linestyle="--", color="black", label="Start Fine-Tuning")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training & Validation Loss (Tuned Baseline)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
best_model = keras.models.load_model(ckpt_path)
test_loss, test_acc = best_model.evaluate(test_ds)
print(f"\n Tuned Test Accuracy: {test_acc:.4f}")


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, f1_score

y_true, y_pred = [], []

for images, labels in test_ds:
    probs = best_model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(probs, axis=1))

print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=class_names,
    yticklabels=class_names,
    cmap="Blues"
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - Tuned Baseline")
plt.show()

In [ ]:
import numpy as np

specificity = []
for i in range(len(cm)):
    tn = np.sum(cm) - (np.sum(cm[i,:]) + np.sum(cm[:,i]) - cm[i,i])
    fp = np.sum(cm[:,i]) - cm[i,i]
    spec = tn / (tn + fp)
    specificity.append(spec)

for cls, spec in zip(class_names, specificity):
    print(f"{cls} Specificity: {spec:.4f}")

In [ ]:

from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

y_true_bin = label_binarize(y_true, classes=range(NUM_CLASSES))

y_prob = []
for images, _ in test_ds:
    y_prob.extend(best_model.predict(images, verbose=0))

y_prob = np.array(y_prob)

plt.figure(figsize=(7,6))
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{class_names[i]} (AUC={roc_auc:.2f})")

plt.plot([0,1],[0,1],'k--')
plt.legend()
plt.title("ROC Curve (One-vs-Rest) - Tuned Baseline")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score

macro_auc = roc_auc_score(
    y_true_bin,
    y_prob,
    average="macro",
    multi_class="ovr"
)

print("Macro Average AUC:", macro_auc)

In [ ]:
def plot_history(h, title_prefix=""):
    hist = h.history
    epochs = range(1, len(hist["loss"]) + 1)

  
    plt.figure(figsize=(7,5))
    plt.plot(epochs, hist["accuracy"], label="Train Accuracy")
    plt.plot(epochs, hist["val_accuracy"], label="Val Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{title_prefix} Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()

    
    plt.figure(figsize=(7,5))
    plt.plot(epochs, hist["loss"], label="Train Loss")
    plt.plot(epochs, hist["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title_prefix} Loss")
    plt.legend()
    plt.grid(True)
    plt.show()


plot_history(history_fe, "Feature Extraction (Tuned)")



In [ ]:

plot_history(history_ft, "Fine-Tuning (Tuned)")

In [ ]:
import optuna.visualization as vis

vis.plot_param_importances(study)


In [ ]:
vis.plot_optimization_history(study)


In [ ]:
vis.plot_parallel_coordinate(study)

In [ ]:
best_model.save("mobilenetv2_tune_baseline.keras")

In [ ]:
best_model.save_weights("mobilenetv2_tune_baseline.weights.h5")